# Assignment Decision Trees & Ensemble Methods

In [1]:
%matplotlib inline
import datetime
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn import linear_model
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
#from skopt import BayesSearchCV
from scipy.stats import randint 
from scipy.stats import uniform
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier


pd.set_option('display.max_rows',1000)
pd.set_option('display.max_columns',1000)

## Part 1. Human Resources

The "Human_Resources.csv" dataset contains various features related to employees, such as satisfaction level, last evaluation, number of projects, average monthly hours, time spent at the company, work accidents, promotions, salary, and whether the employee left the company within the last year (target variable)


In [23]:
# Load the "Human_Resources.csv" dataset into a Pandas DataFrame.
dataset = pd.read_csv('Human_Resources.csv')

# Explore the dataset to understand its structure and characteristics.
dataset.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,work_accident,left,promotion_last_5years
count,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000
mean,0.612834,0.716102,3.803054,201.050337,3.498233,0.144610,0.238083,0.021268
std,0.248631,0.171169,1.232592,49.943099,1.460136,0.351719,0.425924,0.144281
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


In [24]:
dataset.head(10)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,work_accident,left,promotion_last_5years,department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low
5,0.41,0.50,2,153,3,0,1,0,sales,low
6,0.10,0.77,6,247,4,0,1,0,sales,low
7,0.92,0.85,5,259,5,0,1,0,sales,low
8,0.89,1.00,5,224,5,0,1,0,sales,low
9,0.42,0.53,2,142,3,0,1,0,sales,low


In [25]:
#Check for missing values and handle them appropriately.
print(dataset.isnull().sum())

satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
work_accident            0
left                     0
promotion_last_5years    0
department               0
salary                   0
dtype: int64


In [26]:
# Perform feature engineering, such as encoding categorical variables (e.g., salary)
# and handling any necessary data transformations.
dataset = pd.concat([dataset, pd.get_dummies(dataset['department'], prefix='department')], axis=1)
dataset.drop('department', axis=1, inplace=True)

dataset = pd.concat([dataset, pd.get_dummies(dataset['salary'], prefix='salary')], axis=1)
dataset.drop('salary', axis=1, inplace=True)

# Split the data into training and testing sets.
X = dataset.drop('left', axis=1)
y = dataset['left'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=4000, random_state=13)

print(X_test.shape)
print(X_train.shape)

# To avoid conversion warning
X_train = X_train.astype('float64')
X_test = X_test.astype('float64')

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


(4000, 20)
(10999, 20)


In [28]:
# Choose an appropriate machine learning algorithm for binary classification.
# Try the following models: Logistic Regression, Decision Trees
# Random Forest, Adaboost and Gradient Boosting).
log_reg_model = LogisticRegression()

params = {
    'C':[0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'solver':['lbfgs', 'liblinear']
}

log_reg_grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

log_reg_grid_search = log_reg_grid_search.fit(X_train, y_train)

best_log_accuracy = log_reg_grid_search.best_score_
best_log_param = log_reg_grid_search.best_params_

print("Best accuracy (LOG):", best_log_accuracy)
print("Best parameters (LOG):", best_log_param)

y_log_pred = log_reg_grid_search.predict(X_test)

print("\nClassification Report:\n", classification_report(y_test, y_log_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_log_pred))
print("\nAccuracy:", accuracy_score(y_test, y_log_pred) * 100)

Best accuracy (LOG): 0.7861623878622515
Best parameters (LOG): {'C': 1, 'solver': 'lbfgs'}

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.93      0.87      3053
           1       0.60      0.34      0.44       947

    accuracy                           0.79      4000
   macro avg       0.71      0.64      0.65      4000
weighted avg       0.77      0.79      0.77      4000


Confusion Matrix:
 [[2837  216]
 [ 622  325]]

Accuracy: 79.05


In [33]:
# Decision tree
DTclassifier = DecisionTreeClassifier()

params = {
    'class_weight': ['balanced', None],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'min_samples_split': [2, 5, 10, 15, 20],
    'splitter': ['best', 'random']
}

DT_grid_search = GridSearchCV(
    estimator=DTclassifier,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

DT_grid_search = DT_grid_search.fit(X_train, y_train)

best_DT_accuracy = DT_grid_search.best_score_
best_DT_param = DT_grid_search.best_params_

print("Best accuracy (DT):", best_DT_accuracy)
print("Best parameters (DT):", best_DT_param)

y_DT_pred = DT_grid_search.predict(X_test)

print("\nClassification Report:\n", classification_report(y_test, y_DT_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_DT_pred))
print("\nAccuracy:", accuracy_score(y_test, y_DT_pred) * 100)

Best accuracy (DT): 0.9768161561040142
Best parameters (DT): {'class_weight': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'splitter': 'best'}

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99      3053
           1       0.97      0.95      0.96       947

    accuracy                           0.98      4000
   macro avg       0.97      0.97      0.97      4000
weighted avg       0.98      0.98      0.98      4000


Confusion Matrix:
 [[3021   32]
 [  49  898]]

Accuracy: 97.975


In [35]:
# Random Forest
RFclassifier = RandomForestClassifier()

params = {
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

RF_grid_search = GridSearchCV(
    estimator=RFclassifier,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

RF_grid_search = RF_grid_search.fit(X_train, y_train)

best_RF_accuracy = RF_grid_search.best_score_
best_RF_param = RF_grid_search.best_params_

print("Best accuracy (RF):", best_RF_accuracy)
print("Best parameters (RF):", best_RF_param)

y_RF_pred = RF_grid_search.predict(X_test)

print("\nClassification Report:\n", classification_report(y_test, y_RF_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_RF_pred))
print("\nAccuracy:", accuracy_score(y_test, y_RF_pred) * 100)

Best accuracy (RF): 0.9881804952664434
Best parameters (RF): {'class_weight': None, 'min_samples_leaf': 1, 'min_samples_split': 2}

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00      3053
           1       0.99      0.98      0.98       947

    accuracy                           0.99      4000
   macro avg       0.99      0.99      0.99      4000
weighted avg       0.99      0.99      0.99      4000


Confusion Matrix:
 [[3046    7]
 [  23  924]]

Accuracy: 99.25


In [36]:
# Adaboost
ada_classifier = AdaBoostClassifier()

params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

ada_grid_search = GridSearchCV(
    estimator=ada_classifier,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

ada_grid_search = ada_grid_search.fit(X_train, y_train)

best_ada_accuracy = ada_grid_search.best_score_
best_ada_param = ada_grid_search.best_params_

print("Best accuracy (ADA):", best_ada_accuracy)
print("Best parameters (ADA):", best_ada_param)

y_ada_pred = ada_grid_search.predict(X_test)

print("\nClassification Report:\n", classification_report(y_test, y_ada_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_ada_pred))
print("\nAccuracy:", accuracy_score(y_test, y_ada_pred) * 100)

Best accuracy (ADA): 0.9548132622266319
Best parameters (ADA): {'learning_rate': 1.0, 'n_estimators': 200}

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.98      0.97      3053
           1       0.92      0.90      0.91       947

    accuracy                           0.96      4000
   macro avg       0.95      0.94      0.94      4000
weighted avg       0.96      0.96      0.96      4000


Confusion Matrix:
 [[2984   69]
 [  98  849]]

Accuracy: 95.825


In [38]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb_classifier = GradientBoostingClassifier()

params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

gb_grid_search = GridSearchCV(
    estimator=gb_classifier,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

gb_grid_search = gb_grid_search.fit(X_train, y_train)

best_gb_accuracy = gb_grid_search.best_score_
best_gb_param = gb_grid_search.best_params_

print("Best accuracy (GB):", best_gb_accuracy)
print("Best parameters (GB):", best_gb_param)

y_gb_pred = gb_grid_search.predict(X_test)

print("\nClassification Report:\n", classification_report(y_test, y_gb_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_gb_pred))
print("\nAccuracy:", accuracy_score(y_test, y_gb_pred) * 100)

Best accuracy (GB): 0.9809075199470835
Best parameters (GB): {'learning_rate': 0.5, 'n_estimators': 200}

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      3053
           1       0.96      0.97      0.96       947

    accuracy                           0.98      4000
   macro avg       0.98      0.98      0.98      4000
weighted avg       0.98      0.98      0.98      4000


Confusion Matrix:
 [[3018   35]
 [  33  914]]

Accuracy: 98.3


In [42]:
# Determine the most important features that contribute to employee turnover. 
# You can use the feature_importances values computed by a random forest tree
# more information: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.feature_importances_

RFclassifier.fit(X_train, y_train)
importances = RFclassifier.feature_importances_

# Visualize feature importance scores if applicable.
for i in range(len(importances)):
    print(f"Feature {i+1}: {importances[i]}, {X.columns[i]}")

# Find the most important features
most_important_features = np.argsort(importances)[::-1]
print("\nMost important features:")
for i in range(5):
    print(f"Feature {i+1}: {importances[most_important_features[i]]}, {X.columns[most_important_features[i]]}")

Feature 1: 0.30892086087111476, satisfaction_level
Feature 2: 0.12396285269689798, last_evaluation
Feature 3: 0.18720097485405257, number_project
Feature 4: 0.15107271692146484, average_montly_hours
Feature 5: 0.17654120958118363, time_spend_company
Feature 6: 0.010910698924707765, work_accident
Feature 7: 0.0015183627032845962, promotion_last_5years
Feature 8: 0.0016347289238018514, department_IT
Feature 9: 0.0016187207062576306, department_RandD
Feature 10: 0.0017562808620836148, department_accounting
Feature 11: 0.0017758816193103482, department_hr
Feature 12: 0.0019213325056719583, department_management
Feature 13: 0.0012459037874778616, department_marketing
Feature 14: 0.0012640513751059471, department_product_mng
Feature 15: 0.004252417489006786, department_sales
Feature 16: 0.003065252303924535, department_support
Feature 17: 0.004745700705420065, department_technical
Feature 18: 0.005227287313332432, salary_high
Feature 19: 0.007967066971788082, salary_low
Feature 20: 0.0033976

In [43]:
# Compare the different models
from sklearn.metrics import f1_score, precision_score, recall_score

accuracy_scores = {
    'Logistic Regression': accuracy_score(y_test, y_log_pred) * 100,
    'Decision Tree': accuracy_score(y_test, y_DT_pred) * 100,
    'Random Forest': accuracy_score(y_test, y_RF_pred) * 100,
    'AdaBoost': accuracy_score(y_test, y_ada_pred) * 100,
    'Gradient Boosting': accuracy_score(y_test, y_gb_pred) * 100
}

f1_scores = {
    'Logistic Regression': f1_score(y_test, y_log_pred),
    'Decision Tree': f1_score(y_test, y_DT_pred),
    'Random Forest': f1_score(y_test, y_RF_pred),
    'AdaBoost': f1_score(y_test, y_ada_pred),
    'Gradient Boosting': f1_score(y_test, y_gb_pred)
}

precision_scores = {
    'Logistic Regression': precision_score(y_test, y_log_pred),
    'Decision Tree': precision_score(y_test, y_DT_pred),
    'Random Forest': precision_score(y_test, y_RF_pred),
    'AdaBoost': precision_score(y_test, y_ada_pred),
    'Gradient Boosting': precision_score(y_test, y_gb_pred)
}

recall_scores = {
    'Logistic Regression': recall_score(y_test, y_log_pred),
    'Decision Tree': recall_score(y_test, y_DT_pred),
    'Random Forest': recall_score(y_test, y_RF_pred),
    'AdaBoost': recall_score(y_test, y_ada_pred),
    'Gradient Boosting': recall_score(y_test, y_gb_pred)
}

print("Accuracy Scores:")
for model, score in accuracy_scores.items():
    print(f"{model}: {score:.2f}")

print("\nF1 Scores:")
for model, score in f1_scores.items():
    print(f"{model}: {score:.2f}")

print("\nPrecision Scores:")
for model, score in precision_scores.items():
    print(f"{model}: {score:.2f}")

print("\nRecall Scores:")
for model, score in recall_scores.items():
    print(f"{model}: {score:.2f}")

Accuracy Scores:
Logistic Regression: 79.05
Decision Tree: 97.97
Random Forest: 99.25
AdaBoost: 95.83
Gradient Boosting: 98.30

F1 Scores:
Logistic Regression: 0.44
Decision Tree: 0.96
Random Forest: 0.98
AdaBoost: 0.91
Gradient Boosting: 0.96

Precision Scores:
Logistic Regression: 0.60
Decision Tree: 0.97
Random Forest: 0.99
AdaBoost: 0.92
Gradient Boosting: 0.96

Recall Scores:
Logistic Regression: 0.34
Decision Tree: 0.95
Random Forest: 0.98
AdaBoost: 0.90
Gradient Boosting: 0.97


Write down your conclusions: 

- Which is your prefered model and why?
- Which features are the most important ones?
- Which models are suffering from unbalancedness?
- Would you advice the company to use one of these models?


### Conclusions

The best model which I prefer is the Random Forest model. The reason for that is the performance the model delivers. It is the most accurate and has the highest F1, precision and recall scores.

The 5 most important features are:
- satisfaction_level
- number_project
- time_spend_company
- average_montly_hours
- last_evaluation

AdaBoost and Logistic Regression are the models that are suffering from unbalancedness. By analyzing the F1-scores we can see that the Logistic Regression model has the largest difference between class 0 and class 1 which indicates that the model is suffering from unbalancedness.

I would advice the company to use the Random Forest model because it is the most accurate and has the highest F1, precision and recall scores. And also because it is not suffering from unbalancedness.

## Part 2. Energy consumption

Every ten minutes the temperture (in degrees Celcius) and the humidity (in %) of a well insulated house was measured for a couple of months. There is also available weather data from a nearby weather station.
The power consumption of the electric lighting, together with the power consumption of other electrical appliences was recorded during that period same period. 

All measurements can be found in 'Energy_consumption.csv'

The variables have the following meaning:

- date: time year-month-day hour:minute:second
- Appliances: energy use in Wh
- lights: energy use of light fixtures in the house in Wh
- T1: Temperature in kitchen area, in Celsius
- RH_1: Humidity in kitchen area, in %
- T2: Temperature in living room area, in Celsius
- RH_2: Humidity in living room area, in %
- T3: Temperature in laundry room area
- RH_3: Humidity in laundry room area, in %
- T4: Temperature in office room, in Celsius
- RH_4: Humidity in office room, in %
- T5: Temperature in bathroom, in Celsius
- RH_5: Humidity in bathroom, in %
- T6: Temperature outside the building (north side), in Celsius
- RH_6: Humidity outside the building (north side), in %
- T7: Temperature in ironing room , in Celsius
- RH_7: Humidity in ironing room, in %
- T8: Temperature in teenager room 2, in Celsius
- RH_8: Humidity in teenager room 2, in %
- T9: Temperature in parents room, in Celsius
- RH_9: Humidity in parents room, in %
- To: Temperature outside (from Chievres weather station), in Celsius
- Pressure: (from Chievres weather station), in mm Hg 
- RH_out: Humidity outside (from Chievres weather station), in %
- Wind speed: (from Chievres weather station), in m/s
- Visibility: (from Chievres weather station), in km
- Tdewpoint: (from Chievres weather station),
- rv1: Random variable 1, nondimensional
- rv2: Random variable 2, nondimensional


The random variables rv1 and rv2 can be removed from the dataset.


The goal of this assignment is to train a regression model that can predict as precisely as possible the electricity consumption of the appliences from the other variables. 

In [2]:
dataset = pd.read_csv('Energy_consumption.csv')
dataset.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,17.166667,55.20,7.026667,84.256667,17.200000,41.626667,18.2,48.900000,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,17.166667,55.20,6.833333,84.063333,17.200000,41.560000,18.2,48.863333,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,17.166667,55.09,6.560000,83.156667,17.200000,41.433333,18.2,48.730000,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,17.166667,55.09,6.433333,83.423333,17.133333,41.290000,18.1,48.590000,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,17.200000,55.09,6.366667,84.893333,17.200000,41.230000,18.1,48.590000,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


1. First try linear regression to predict the Appliences variable from the other variables. Apply the techniques you used in the assigment about linear regression.
2. Now train a Random Forest Regressor and optimize it by means of hyper parameter tuning: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html


You get all the freedom to use the techniques and tricks you want. Your only goals is to achieve the best R²-score on a test set consisting of 5000 samples. It might be useful to use the date and time of the day as features.

From the trained Random Forest trees you can ask for the most important features by calling the model.feature_importances_.
Give the top 5 most important features. Do they make sense? Explain.


In [3]:
# Convert and split date into month, day, hour
dataset['date'] = pd.to_datetime(dataset.date)
dataset.insert(0,'month', dataset['date'].dt.month)
dataset.insert(0,'day',  dataset['date'].dt.weekday)
dataset.insert(0,'hour', dataset['date'].dt.hour)
dataset['month'] = dataset['month'].apply(lambda x: calendar.month_name[x])
dataset['day'] = dataset['day'].apply(lambda x: calendar.day_name[x])

dataset.drop('date',axis=1,inplace=True)

dataset.head()

,hour,day,month,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,17,Monday,January,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,17.166667,55.20,7.026667,84.256667,17.200000,41.626667,18.2,48.900000,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,17,Monday,January,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,17.166667,55.20,6.833333,84.063333,17.200000,41.560000,18.2,48.863333,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,17,Monday,January,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,17.166667,55.09,6.560000,83.156667,17.200000,41.433333,18.2,48.730000,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,17,Monday,January,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,17.166667,55.09,6.433333,83.423333,17.133333,41.290000,18.1,48.590000,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,17,Monday,January,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,17.200000,55.09,6.366667,84.893333,17.200000,41.230000,18.1,48.590000,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [12]:
# Remove rv1 and rv2 from the dataset


In [13]:
# One-hot encoding of the categorical features


In [14]:
# Split into features and targets


In [15]:
# Split into training set and test set


In [16]:
# MinMax scaling


In [17]:
# Linear regression


In [18]:
# Model optimization and hyperparameter tuning of the linear regression model. You are allowed to use features expansion (hihger order features)


In [ ]:
# Random forest regressor. You can also try higher order features. You are allowed to use features expansion (hihger order features)


In [19]:
# Most important features + conclusions


## Part 3 - Bank

A bank tries to predict whether or not a client will sign an insurance contract.
The file *bank.csv* contains data from over 4000 clients.
The features are the following:
- age:  age of the client.
- job:  job type the client has.
- marital:  marital status.
- education:  type of diploma.
- default: whether or not the client has been declared bankrupt.
- balance: amount of money on the account.
- housing:  whether or not the client has a housing loan.
- loan:  whether or not the client has a personal loan.
- contact: type of communication with the client.
- day: day of the last contact with the client.
- month: month of the last contact with the client.
- duration: duration of the last contact. Cannot be used to train on. Has to be discarded from the dataset.
- campaign: number of previous contacts with the client.
- pdays: number of days since the previous contact. -1 means that the client was not contacted before.
- previous:  number of previous contacts with the client.
- poutcome: outcome of a previous campaign.

The target is the y-column. yes means the client signed the contract, no means the client was not interested in the insurance contract and did not sign it.


In [20]:
dataset = pd.read_csv('bank.csv',delimiter=';')
dataset.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


## Preprocessing

In [4]:
# check for consitency
dataset.describe()

,age,balance,day,duration,campaign,pdays,previous
count,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000
mean,41.170095,1422.657819,15.915284,263.961292,2.793630,39.766645,0.542579
std,10.576211,3009.638142,8.247667,259.856633,3.109807,100.121124,1.693562
min,19.000000,-3313.000000,1.000000,4.000000,1.000000,-1.000000,0.000000
25%,33.000000,69.000000,9.000000,104.000000,1.000000,-1.000000,0.000000
50%,39.000000,444.000000,16.000000,185.000000,2.000000,-1.000000,0.000000
75%,49.000000,1480.000000,21.000000,329.000000,3.000000,-1.000000,0.000000
max,87.000000,71188.000000,31.000000,3025.000000,50.000000,871.000000,25.000000


Controleer of de data al dan niet gebalanceerd is. Wat zijn de conclusies?

In [5]:
# check for unbalancedness of the dataset


In [6]:
# Remove the duration column from the dataset


# replace label y: no -> 0 and yes -> 1



In [7]:
# One hot encoding of categorical features


Create a trainig set and test set.
Make sure you have 1000 samples in the test set and use a random_state = 0.

In [8]:
# Split into features and targets


# Split into training set and test set


# MinMax scaler normalisation or standard scaler normalization





## Training of the classifiers

Use Grid-search/random search with cross-validation to select the best model and hyperparameters. 

 
**Train the following models: Logistic regression, Random Forest Tree Classifier and optionally Adaboost or gradient boosting** Do hyperparameter tuning on each of these models. Also change the cross-validation parameter K. Don't forget to scale the data. ** Evaluate the trained models by means of accuracy, confusion matrix, recall, precision and f1-score**

Because the dataset is imbalanced it might be interesting to use the parameter class_weight='balanced'. This hyperparemter is supported by most of the classification models. 
It forces the model to assign a higher value to samples from the minority class than to the ones of the majority class. Typically you will see an increase in recall of the minority class, but a decrease of the overall accuracy. Explain why this is the case.



In [44]:
# Logistic regression


In [42]:
# Support Vector machine



In [43]:
# Random forest trees



In [45]:
# Adaboost - optional



In [46]:
# Adaboost with logistic regression classifier - optional



In [47]:
# Gradient boosting - optional





## Optimization

- Are there features that may be discarded? If so, which ones? You can find the features importances if you use random forest trees by calling **tree.feature_importances_**.
- What are the three most important features?
- Retrain the models with the 10 most important features.